# 03 — Gold/Blue J-Lens × Logit Lens sweep

**Goal.** Apply the fixed base-model J-Lens and vanilla Logit Lens to the
same saved base/Gold/Blue sequences over published standard and direct
prompts.

The long job runs as a resumable script in `tmux`, as required by the project
protocol. This notebook prepares the command, monitors immutable per-sequence
artifacts, and inspects the resulting Parquet file.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import behavior_dataframe, require_behavior_approval
from src.jlens_sanity import require_sanity_approval

paths, config = open_run(RUN_ID)
require_behavior_approval(paths, config)
require_sanity_approval(paths, config)
behavior = behavior_dataframe(paths)
required_prompts = config["prompts"]["groups"]["lens_sweep"]
required_conditions = config["behavior"]["conditions"]
display(behavior[
    behavior["prompt_id"].isin(required_prompts)
    & behavior["condition"].isin(required_conditions)
][["prompt_id", "prompt_type", "condition", "own_secret_leaked", "generation_token_count"]])


## Measurement plan

- Full layer coverage from the public J-Lens checkpoint.
- Positions: last 10 rendered-input tokens plus every generated token.
- Cheap Gold/Blue candidate logits at every measured layer × position.
- Full-vocabulary ranks/top-10 at every layer for the last input token,
  first generated token, and five generated quantiles.
- Full-vocabulary ranks/top-10 at every position for anchor layers 32, 48, 51.
- Prior confirmatory band: layers 37–58. The full sweep remains exploratory;
  the band summary avoids selecting the best layer after seeing results.
- Outputs containing their own secret remain saved but are excluded from the
  headline metric.

Layer 48 is an a-priori anchor because the pinned Qwen3.6 source reports its
best Activation-Oracle Taboo readout at 75% depth; layer 32 is that source's
mid-stack failure/control, and 51 is a fixed nearby late-stack diagnostic.
This motivates where to inspect, but it does not imply that an AO optimum must
be a J-Lens optimum.

The user-requested restriction to existing prompts means this phase uses the
published standard and direct splits only. Base and the other adapter provide
the main controls; a newly authored unrelated-topic or leakage-positive prompt
would be a separate later condition and is intentionally not introduced here.


## Why this comparison follows the references

The original Taboo work ([2505.14352](https://arxiv.org/abs/2505.14352)) and
the larger secret-elicitation benchmark
([2510.01070](https://arxiv.org/abs/2510.01070)) make Logit Lens the direct
white-box baseline and keep direct black-box attacks behaviorally distinct.
The activation-oracle confidence study
([2605.26045](https://arxiv.org/abs/2605.26045)) shows why scoring a known
candidate set is easier: Gold/Blue candidate metrics are therefore auxiliary,
while full-vocabulary ranks/top-k are required. The natural-censorship study
([2603.05494](https://arxiv.org/abs/2603.05494)) motivates later matched
censorship controls but does not turn this synthetic two-adapter pilot into a
natural-censorship result.


In [ ]:
display(config["readout"])

missing = []
for prompt_id in required_prompts:
    for condition in required_conditions:
        rows = behavior[
            behavior["prompt_id"].eq(prompt_id)
            & behavior["condition"].eq(condition)
        ]
        if len(rows) != 1:
            missing.append((prompt_id, condition, len(rows)))
assert not missing, f"Missing or duplicate behavior rows: {missing}"


## Start the long job in tmux

The Jupyter process cannot keep its 27B model while a second copy loads in the
tmux worker. After all earlier artifacts are saved, deliberately approve the
GPU handoff below. It clears only in-memory model/lens objects, does not restart
the kernel, and records free HBM. The model can be reconstructed from the
pinned run if needed.


In [ ]:
APPROVE_GPU_HANDOFF = False  # Change deliberately after notebooks 01–02 are complete.
MIN_FREE_GIB = 65

if not APPROVE_GPU_HANDOFF:
    raise RuntimeError("Approve the GPU handoff before starting the tmux worker.")

from src.model_session import release_session

handoff = release_session(paths)
display(handoff)
if handoff.get("cuda_free_gib", 0) < MIN_FREE_GIB:
    raise RuntimeError(
        f"Only {handoff.get('cuda_free_gib', 0):.1f} GiB is free; "
        "inspect other GPU processes before launching the sweep."
    )


Run the printed command in a RunPod terminal. It never overwrites completed
`prompt × condition` cell files. If the process stops, run the same command
again with the same `RUN_ID`.


In [ ]:
session_name = "jlens-" + "".join(
    character if character.isalnum() else "-" for character in RUN_ID
)[-40:]
log_path = PROJECT_ROOT / "logs" / f"{RUN_ID}_lens_sweep.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
import shlex

inner_command = (
    f"cd {shlex.quote(str(PROJECT_ROOT))} && source .venv/bin/activate && "
    f"python scripts/run_gold_blue_sweep.py --run-id {shlex.quote(RUN_ID)} "
    f"2>&1 | tee {shlex.quote(str(log_path))}"
)
command = (
    f"tmux new-session -d -s {shlex.quote(session_name)} "
    f"{shlex.quote(inner_command)}"
)
print(command)
print("Log:", log_path)


## Non-destructive progress monitor

This cell only counts completed atomic files and displays the tail of the log.
It does not interrupt the process or kernel.


In [ ]:
cell_files = sorted((paths.lens_dir / "cells").glob("*.jsonl"))
expected_sequences = len(required_prompts) * len(required_conditions)
print(f"completed sequences: {len(cell_files)} / {expected_sequences}")
for file in cell_files[-10:]:
    print(file.name, f"{file.stat().st_size / 2**20:.1f} MiB")
if log_path.exists():
    print("\n--- log tail ---")
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-40:]))


## Inspect completed export

Run only after the monitor reports all sequences and the script writes
`lens_readouts.parquet`.


In [ ]:
parquet_path = paths.result_dir / "lens_readouts.parquet"
assert parquet_path.exists(), "Sweep/export is not complete yet."
sample = pd.read_parquet(parquet_path).head(50)
print("Parquet:", parquet_path, f"{parquet_path.stat().st_size / 2**20:.1f} MiB")
display(sample)
